# Etapa 5 — Simulación completa y esquemas de actualización

**Objetivo:** comprobar que el modelo integrado funcione de manera síncrona y asíncrona. Se verifica el avance temporal, la ausencia de superposiciones, la conservación de la población y la capacidad de las salidas.

**Antes de ejecutar:** completar `_hacer_paso_movimiento()` y `avanzar()` de `Simulacion`.

## Pruebas 1 y 2. Avance del tiempo y ocupación

Se ejecuta **un paso** para verificar que el reloj pase de 0 a 1. Luego se realizan dos simulaciones y, **después de cada paso**, se comprueba que no existan dos agentes no evacuados en la misma celda.

In [ ]:
import matplotlib.pyplot as plt
from modelo import Configuracion, Simulacion
from visualizacion import (
    mostrar_estado, graficar_curvas_evacuacion,
    graficar_ocupacion_acumulada, mostrar_en_notebook,
)

base = Configuracion(n_agentes=45, semilla=23, pasos_maximos=420)

# Prueba 1: avanzar() debe ejecutar y registrar exactamente un paso.
simulacion_corta = Simulacion(base.copiar_con(n_agentes=20))
assert simulacion_corta.tiempo == 0
assert simulacion_corta.avanzar()
assert simulacion_corta.tiempo == 1
assert simulacion_corta.historial[-1]["paso"] == 1

# Prueba 2: comprobar que no haya posiciones repetidas después de cada paso.
simulaciones = []
for esquema in ("sincrona", "asincrona"):
    simulacion = Simulacion(base.copiar_con(actualizacion=esquema))
    while simulacion.avanzar():
        posiciones = [a.posicion() for a in simulacion.agentes if not a.evacuado]
        assert len(posiciones) == len(set(posiciones))
    simulaciones.append(simulacion)
    print(esquema, simulacion.indicadores())

sincrona, asincrona = simulaciones
figura = mostrar_estado(asincrona)
plt.show()
print("Pruebas 1 y 2: tiempo y ocupación correctos.")

## Prueba 3. Conservación y capacidad de las salidas

Para estas dos realizaciones se espera la evacuación completa de 45 agentes. En **todos los registros** debe cumplirse `restantes + evacuados = 45`, y entre dos pasos consecutivos no deben evacuar más de **cuatro personas** (dos celdas por salida).

In [ ]:
for simulacion in (sincrona, asincrona):
    assert simulacion.indicadores()["fraccion_evacuada"] == 1

    # Conservar la cantidad total de personas en cada paso.
    for estado in simulacion.historial:
        assert estado["restantes"] + estado["evacuados"] == 45

    # Comparar nuevos evacuados entre pasos consecutivos.
    for anterior, posterior in zip(simulacion.historial, simulacion.historial[1:]):
        nuevas_evacuaciones = posterior["evacuados"] - anterior["evacuados"]
        assert 0 <= nuevas_evacuaciones <= 4

print("Prueba 3: conservación y capacidad correctas.")

## Visualizaciones

Las curvas muestran la fracción evacuada durante la simulación. El mapa de ocupación acumulada indica cuánto tiempo permanecieron ocupadas las celdas. **Estos gráficos son para observar el comportamiento, no pruebas automáticas adicionales.**

In [ ]:
# Comparar curvas de evacuación síncrona y asíncrona.
figura = graficar_curvas_evacuacion(
    [sincrona, asincrona], ["Síncrona", "Asíncrona"]
)
plt.show()

# Mostrar las zonas de mayor permanencia de agentes.
figura = graficar_ocupacion_acumulada(asincrona)
plt.show()

## Animación opcional

Descomentá la línea para ver los desplazamientos en la notebook, sin guardar archivos.

In [ ]:
from IPython.display import display
# display(mostrar_en_notebook(asincrona, intervalo_ms=120))

**Para discutir:** ¿cómo pueden generarse congestiones diferentes si las reglas individuales son las mismas?